# NB14 — Final PCA/LDA Descriptive Projection

Public, output-stripped reproducibility notebook for the manuscript's combined PCA/LDA descriptive figure. It does not estimate predictive performance and does not perform calibration.

**Recommended runtime:** CPU.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
DATASET = ROOT/'01_DATA'/'INIAP_Dataset.xlsx'
OUT = ROOT/'03_RESULTS'/'NB14_FINAL_PCA_LDA'
FIG = ROOT/'05_FIGURES'/'FINAL_PUBLICATION'
OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)

In [ ]:
df = pd.read_excel(DATASET).rename(columns={'AspectRation':'AspectRatio'})
TARGET='Class'
X=df.drop(columns=[TARGET])
y=df[TARGET].astype(str)
assert len(df)==4000 and y.nunique()==4

Xs=StandardScaler().fit_transform(X)
P=PCA(n_components=2).fit(Xs)
PC=P.transform(Xs)
le=LabelEncoder(); ye=le.fit_transform(y)
L=LinearDiscriminantAnalysis(n_components=2).fit(Xs,ye)
LD=L.transform(Xs)
classes=sorted(y.unique())

fig,ax=plt.subplots(1,2,figsize=(12.8,5.3))
for cls in classes:
    m=(y.values==cls); ax[0].scatter(PC[m,0],PC[m,1],s=17,alpha=.48,linewidths=0,label=cls)
ax[0].set_xlabel(f'PC1 ({P.explained_variance_ratio_[0]*100:.1f}% variance)')
ax[0].set_ylabel(f'PC2 ({P.explained_variance_ratio_[1]*100:.1f}% variance)')
ax[0].set_title('(a) Principal Component Analysis'); ax[0].grid(alpha=.15)
ax[1].set_prop_cycle(None)
for cls in classes:
    m=(y.values==cls); ax[1].scatter(LD[m,0],LD[m,1],s=17,alpha=.48,linewidths=0,label=cls)
ax[1].set_xlabel(f'LD1 ({L.explained_variance_ratio_[0]*100:.1f}% discriminant variance)')
ax[1].set_ylabel(f'LD2 ({L.explained_variance_ratio_[1]*100:.1f}% discriminant variance)')
ax[1].set_title('(b) Linear Discriminant Analysis'); ax[1].grid(alpha=.15)
h,l=ax[0].get_legend_handles_labels(); fig.legend(h,l,loc='lower center',ncol=4,frameon=False,bbox_to_anchor=(.5,-.01))
plt.tight_layout(rect=[0,.08,1,1])
stem=FIG/'Figure3_PCA_LDA_Projection_PUBLICATION'
plt.savefig(str(stem)+'.png',dpi=600,bbox_inches='tight'); plt.savefig(str(stem)+'.pdf',bbox_inches='tight'); plt.show()

pd.DataFrame({'PC1':PC[:,0],'PC2':PC[:,1],TARGET:y}).to_csv(OUT/'pca_scores_pc1_pc2.csv',index=False)
pd.DataFrame({'LD1':LD[:,0],'LD2':LD[:,1],TARGET:y}).to_csv(OUT/'lda_scores_ld1_ld2.csv',index=False)